In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import sys
project_path = "/content/drive/MyDrive/Flight-Safety"
os.chdir(project_path)
sys.path.append(project_path)

In [3]:
import tensorflow as tf
print("GPU Available:", tf.config.list_physical_devices('GPU'))

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
!pip install import-ipynb

In [5]:
import import_ipynb
from load_process_data import load_data, preprocess_data, shuffle_data
from models import BaseGRU, MultiHeadCnnRnn
from keras.regularizers import l2
from model_eval import cross_validate
from visualize import visualize_accuracies

In [6]:
# Define hyperparameters

input_shape = (81, 23)  # 81 timesteps with 23 features
num_folds = 5 # Cross-validation
epochs = 30

# GRU hyperparameters

learning_rate = 0.002
dropout = 0.1
recurrent_dropout = 0.1
kernel_regularizer = l2(0.01)  # Regularization strength for kernel
recurrent_regularizer = l2(0.01)  # Regularization strength for recurrent connections

# CNN hyperparameters

kernel_sizes = [8, 5, 3]
filters = [16, 32, 64]
learning_rate = 0.001
weight_decay = 0.01

# Create models with regularization

GRUModel = BaseGRU(input_shape=input_shape,
                learning_rate=learning_rate,
                dropout=dropout,
                recurrent_dropout=recurrent_dropout,
                kernel_regularizer=kernel_regularizer,
                recurrent_regularizer=recurrent_regularizer)

MultiHeadModel = MultiHeadCnnRnn(input_shape=input_shape,
                                 kernel_sizes=kernel_sizes,
                                 filters=filters,
                                 learning_rate=learning_rate,
                                 weight_decay=weight_decay,
                                 dropout=dropout,
                                 recurrent_dropout=recurrent_dropout,
                                 kernel_regularizer=kernel_regularizer,
                                 recurrent_regularizer=recurrent_regularizer)



/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Feature amount: 23


In [7]:
data_path = "/content/drive/MyDrive/Flight-Safety/data/"

train_y_list, test_y_list, train_X_list, test_X_list = load_data(data_path)

# Preprocessing data (reshape + standardization)

train_x_list_filtered, test_x_list_filtered = preprocess_data(train_X_list, test_X_list)
print(train_x_list_filtered[0].shape)
print(train_y_list[0].shape)
print(test_x_list_filtered[0].shape)
print(test_y_list[0].shape)

Loading data ...
Data shapes: 
train_x_list: (5,)
training data within each fold: (16063, 81, 25)
train_y_list: (5,)
training label within each fold: (16063,)
Preprocessing data ...
(16063, 81, 23)
(16063,)
(4016, 81, 23)
(4016,)


In [8]:
# Shuffling data
train_x_list_shuffled = []
train_y_list_shuffled = []

for i, fold in enumerate(train_x_list_filtered):
    train_x_fold, train_y_fold = shuffle_data(fold, train_y_list[i])
    train_x_list_shuffled.append(train_x_fold)
    train_y_list_shuffled.append(train_y_fold)


In [ ]:
# BaseGRU training and testing

train_accuracies, test_accuracies = cross_validate(GRUModel, num_folds, train_x_list_filtered,
                                                   train_y_list, test_x_list_filtered, test_y_list)


Testing on Base GRU class: 
Training on fold 1/5
Epoch 1/10
502/502 ━━━━━━━━━━━━━━━━━━━━ 140s 251ms/step - accuracy: 0.6405 - loss: 1.0262 - val_accuracy: 0.7517 - val_loss: 0.5942
Epoch 2/10
502/502 ━━━━━━━━━━━━━━━━━━━━ 134s 243ms/step - accuracy: 0.7525 - loss: 0.6032 - val_accuracy: 0.7781 - val_loss: 0.5506
Epoch 3/10
502/502 ━━━━━━━━━━━━━━━━━━━━ 139s 237ms/step - accuracy: 0.7724 - loss: 0.5772 - val_accuracy: 0.8190 - val_loss: 0.5055
Epoch 4/10
502/502 ━━━━━━━━━━━━━━━━━━━━ 123s 244ms/step - accuracy: 0.7910 - loss: 0.5506 - val_accuracy: 0.7727 - val_loss: 0.5627
Epoch 5/10
502/502 ━━━━━━━━━━━━━━━━━━━━ 142s 244ms/step - accuracy: 0.7950 - loss: 0.5418 - val_accuracy: 0.7961 - val_loss: 0.5257
Epoch 6/10
502/502 ━━━━━━━━━━━━━━━━━━━━ 142s 244ms/step - accuracy: 0.8020 - loss: 0.5337 - val_accuracy: 0.6703 - val_loss: 0.6770
Epoch 7/10
502/502 ━━━━━━━━━━━━━━━━━━━━ 141s 242ms/step - accuracy: 0.8077 - loss: 0.5285 - val_accuracy: 0.7403 - val_loss: 0.5887
Epoch 8/10
206/502 ━━━━━━━━